# 004 - Модуль построения feature dataset

Notebook вокруг `research/features/build_features_dataset.py`.

Структура:
1. полный исходный код feature-модуля,
2. краткое объяснение архитектуры,
3. пример вызова для `target parquet -> features parquet`.


In [1]:
from pathlib import Path


# Ищем корень репозитория независимо от того, из какой папки запущен notebook.
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "research").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Не удалось найти корень репозитория с папками research/ и notebooks/")


ROOT = find_repo_root(Path.cwd())
MODULE_PATH = ROOT / "research" / "features" / "build_features_dataset.py"
print(MODULE_PATH)
print(MODULE_PATH.read_text(encoding="utf-8"))


D:\tumar\okx-hft-research\research\features\build_features_dataset.py
# flake8: noqa
"""Build leakage-safe feature datasets from reconstructed order book data.

This module assumes that:
- the input DataFrame already contains target columns that must be preserved,
- all features are built using current and past information only,
- features should be scale-invariant with respect to the absolute BTC price.

The implementation therefore avoids absolute price levels as model features and
uses relative, normalized, and dynamic microstructure signals.
"""

from __future__ import annotations

import argparse
import logging
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from research.utils.tick_size import infer_tick_size

LOGGER = logging.getLogger(__name__)

BASE_COLUMNS = [
    "ts_event",
    "inst_id",
    "anchor_snapshot_ts",
    "reconstruction_mode",
    "mid_px",
    "spread_px",
]

BOOK_COLUMNS = [
    *[f"bid_px_{i:02d}" for i in range(1, 1

## Ключевые решения

- Все lookback-окна считаются по реальному времени `ts_event` через `np.searchsorted`, а не по фиксированному числу строк.
- В признаки не подмешиваются future target-колонки: расчёты используют только текущее и прошлое состояние стакана.
- Абсолютные уровни цены не используются как features; вместо этого берутся расстояния в тиках, нормализованные объёмы, imbalance и динамика.
- Модуль устроен по группам фич: `price`, `depth`, `imbalance`, `order_flow`, `regime`.
- Для полного feature engineering нужны и `target columns`, и сырой стакан `bid_* / ask_*`. Если target parquet уже сохранён без стакана, его нужно сначала объединить с raw parquet по ключевым колонкам.
- Через `keep_raw_book=False` можно убрать сырые book-колонки и оставить более компактный ML-ready dataset.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from research.features.build_features_dataset import build_features_dataset, summarize_features
from research.utils.tick_size import infer_tick_size


# Ищем корень репозитория независимо от текущей рабочей директории kernel.
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "research").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Не удалось найти корень репозитория с папками research/ и notebooks/")


ROOT = find_repo_root(Path.cwd())
RECONSTRUCTED_DIR = ROOT / "data" / "reconstructed" / "BTC-USDT-SWAP" / "grid100ms"
TARGETS_PARQUET_PATH = ROOT / "outputs" / "targets" / "btc_usdt_swap_targets_dataset.parquet"
OUTPUT_PATH = ROOT / "outputs" / "features" / "btc_usdt_swap_features_dataset.parquet"

# Базовые параметры сборки feature dataset.
LOOKBACK_SEC = 30
KEEP_RAW_BOOK = False

BOOK_REQUIRED = [
    *[f"bid_px_{i:02d}" for i in range(1, 11)],
    *[f"bid_sz_{i:02d}" for i in range(1, 11)],
    *[f"ask_px_{i:02d}" for i in range(1, 11)],
    *[f"ask_sz_{i:02d}" for i in range(1, 11)],
]
base_join_candidates = ["ts_event", "inst_id", "anchor_snapshot_ts", "reconstruction_mode", "mid_px", "spread_px"]

# Ищем parquet-файлы восстановленного стакана по дням.
raw_parquet_files = sorted(RECONSTRUCTED_DIR.rglob("*.parquet"))
if not raw_parquet_files:
    raise FileNotFoundError(f"Не найдено parquet-файлов в {RECONSTRUCTED_DIR}")

# Берём schema первого parquet и убеждаемся, что это действительно order book snapshots.
raw_schema_cols = pq.read_schema(raw_parquet_files[0]).names
missing_book_cols = [col for col in BOOK_REQUIRED if col not in raw_schema_cols]
if missing_book_cols:
    raise ValueError(
        "Parquet в reconstructed/.../grid100ms не содержит полный стакан. "
        f"Например отсутствуют: {missing_book_cols[:8]}"
    )

# Загружаем только нужные колонки из reconstructed parquet-ов.
raw_columns_to_read = [col for col in base_join_candidates + BOOK_REQUIRED if col in raw_schema_cols]
df_raw = pd.concat(
    [pd.read_parquet(path, columns=raw_columns_to_read) for path in raw_parquet_files],
    ignore_index=True,
)

# Из target parquet читаем только ключи и target-only колонки.
target_schema_cols = pq.read_schema(TARGETS_PARQUET_PATH).names
join_keys = [
    col
    for col in base_join_candidates
    if col in df_raw.columns and col in target_schema_cols
]
target_only_cols = [col for col in target_schema_cols if col not in df_raw.columns]
df_targets = pd.read_parquet(TARGETS_PARQUET_PATH, columns=join_keys + target_only_cols)

# Дедупликация и стабильный one-to-one матч по occurrence внутри одинаковых ключей.
df_raw = df_raw.sort_values(join_keys).reset_index(drop=True)
df_targets = df_targets.sort_values(join_keys).reset_index(drop=True)

df_raw_keys = df_raw[join_keys].copy()
df_raw_keys["_raw_row_id"] = np.arange(len(df_raw_keys), dtype=np.int64)
df_raw_keys["_merge_occurrence"] = df_raw_keys.groupby(join_keys, dropna=False).cumcount()

df_targets_keys = df_targets[join_keys].copy()
df_targets_keys["_merge_occurrence"] = df_targets_keys.groupby(join_keys, dropna=False).cumcount()

merge_keys = join_keys + ["_merge_occurrence"]
df_match = df_raw_keys.merge(
    pd.concat([df_targets_keys[merge_keys], df_targets[target_only_cols]], axis=1),
    on=merge_keys,
    how="inner",
    validate="one_to_one",
)

# Собираем итоговый dataframe без фрагментации: raw block + target block одним concat.
df_raw_matched = df_raw.iloc[df_match["_raw_row_id"].to_numpy()].reset_index(drop=True)
df_target_block = df_match[target_only_cols].reset_index(drop=True)
df_for_features = pd.concat([df_raw_matched, df_target_block], axis=1).copy()

book_cols_check = {
    col: (col in df_for_features.columns)
    for col in ["bid_px_01", "bid_px_10", "bid_sz_01", "bid_sz_10", "ask_px_01", "ask_px_10", "ask_sz_01", "ask_sz_10"]
}
print("RECONSTRUCTED_DIR:", RECONSTRUCTED_DIR)
print("reconstructed parquet files:", len(raw_parquet_files))
print("raw shape:", df_raw.shape)
print("targets selected shape:", df_targets.shape)
print("matched rows:", df_match.shape)
print("final df_for_features shape:", df_for_features.shape)
print("join keys:", merge_keys)
print("target_only_cols:", len(target_only_cols))
print("book cols in df_for_features:", book_cols_check)

missing_after_merge = [col for col in BOOK_REQUIRED if col not in df_for_features.columns]
if missing_after_merge:
    raise ValueError(
        "После merge в df_for_features всё ещё нет book columns. "
        f"Например отсутствуют: {missing_after_merge[:8]}"
    )

tick_size = infer_tick_size(df_for_features, price_col="mid_px").tick_size

# Строим dataset с фичами поверх уже существующих target columns.
df_features = build_features_dataset(
    df_for_features,
    tick_size=tick_size,
    lookback_sec=LOOKBACK_SEC,
    keep_raw_book=KEEP_RAW_BOOK,
)

# Смотрим компактную сводку по группам фич, NaN и базовой статистике.
summary = summarize_features(df_features)
display(df_features.head())
display(pd.Series({k: len(v) for k, v in summary["feature_columns_by_group"].items()}, name="n_features"))
display(summary["nan_pct"].head(20))
display(summary["basic_stats"].head(20))

# Сохраняем результат в parquet.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_features.to_parquet(OUTPUT_PATH, index=False)
print("Сохранено в", OUTPUT_PATH)


RECONSTRUCTED_DIR: D:\tumar\okx-hft-research\data\reconstructed\BTC-USDT-SWAP\grid100ms
reconstructed parquet files: 72
raw shape: (2592072, 46)
targets selected shape: (2590252, 166)
matched rows: (2590252, 168)
final df_for_features shape: (2590252, 206)
join keys: ['ts_event', 'inst_id', 'anchor_snapshot_ts', 'reconstruction_mode', 'mid_px', 'spread_px', '_merge_occurrence']
target_only_cols: 160
book cols in df_for_features: {'bid_px_01': True, 'bid_px_10': True, 'bid_sz_01': True, 'bid_sz_10': True, 'ask_px_01': True, 'ask_px_10': True, 'ask_sz_01': True, 'ask_sz_10': True}


D:\tumar\okx-hft-research\research\features\build_features_dataset.py:614: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = features[col]
D:\tumar\okx-hft-research\research\features\build_features_dataset.py:614: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = features[col]
D:\tumar\okx-hft-research\research\features\build_features_dataset.py:614: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

,ts_event,inst_id,anchor_snapshot_ts,reconstruction_mode,mid_px,spread_px,long_max_favorable_excursion_ticks_within_30s,long_max_adverse_excursion_ticks_within_30s,short_max_favorable_excursion_ticks_within_30s,short_max_adverse_excursion_ticks_within_30s,...,ask_size_removed_5s,net_bid_flow_5s,net_ask_flow_5s,order_flow_imbalance_5s,update_intensity_1s,update_intensity_5s,spread_regime,volatility_regime,update_regime,seconds_since_large_move
0,2026-03-23 00:00:00+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,1642.0,0.0,0.0,1642.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,<NA>,NaN
1,2026-03-23 00:00:00.100000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,1642.0,0.0,0.0,1642.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,<NA>,NaN
2,2026-03-23 00:00:00.200000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67830.85,0.1,1642.0,0.0,0.0,1642.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,<NA>,NaN
3,2026-03-23 00:00:00.300000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67840.65,0.1,1544.0,0.0,0.0,1544.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,<NA>,NaN
4,2026-03-23 00:00:00.400000+00:00,BTC-USDT-SWAP,2026-03-22 23:59:47.104000+00:00,grid_100ms,67843.85,0.1,1512.0,0.0,0.0,1512.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,<NA>,<NA>,NaN


price         12
depth         58
imbalance     13
order_flow    23
regime         4
Name: n_features, dtype: int64

order_flow_imbalance_1s    0.074742
order_flow_imbalance_3s    0.068333
order_flow_imbalance_5s    0.064936
realized_vol_30s           0.011582
volatility_regime          0.011582
mid_return_10s             0.003861
realized_vol_10s           0.003861
imbalance_10_delta_5s      0.001930
ask_size_added_5s          0.001930
ask_size_removed_5s        0.001930
bid_size_removed_5s        0.001930
bid_size_added_5s          0.001930
imbalance_1_delta_5s       0.001930
imbalance_3_delta_5s       0.001930
net_bid_flow_5s            0.001930
realized_vol_5s            0.001930
imbalance_5_delta_5s       0.001930
mid_return_5s              0.001930
net_ask_flow_5s            0.001930
update_regime              0.001930
dtype: float64

,mean,std,min,median,max
ask_dist_10_ticks,15.292863,7.501749,9.5,11.5,623.5
ask_dist_1_ticks,0.507324,0.621934,0.5,0.5,232.0
ask_dist_2_ticks,1.581628,1.193219,1.5,1.5,501.5
ask_dist_3_ticks,3.954041,3.274647,2.5,2.5,518.5
ask_dist_4_ticks,5.694006,4.088109,3.5,4.5,535.5
ask_dist_5_ticks,7.447324,4.822786,4.5,5.5,554.5
ask_dist_6_ticks,9.092363,5.459593,5.5,6.5,564.5
ask_dist_7_ticks,10.682968,6.031529,6.5,8.5,617.5
ask_dist_8_ticks,12.267794,6.557651,7.5,9.5,619.5
ask_dist_9_ticks,13.791231,7.054393,8.5,10.5,622.5


Сохранено в D:\tumar\okx-hft-research\outputs\features\btc_usdt_swap_features_dataset.parquet


In [3]:
# Пример запуска модуля из командной строки.
cli_example = r'''
python -m research.features.build_features_dataset \
  --parquet-path outputs/targets/btc_usdt_swap_targets_dataset.parquet \
  --output-path outputs/features/btc_usdt_swap_features_dataset.parquet \
  --lookback-sec 30
'''
print(cli_example)



python -m research.features.build_features_dataset \
  --parquet-path outputs/targets/btc_usdt_swap_targets_dataset.parquet \
  --output-path outputs/features/btc_usdt_swap_features_dataset.parquet \
  --lookback-sec 30

